In [3]:
# ============================================================
# IMPORT THƯ VIỆN
# ============================================================
import numpy as np
import pandas as pd
import os, glob
import matplotlib.pyplot as plt
import seaborn as sns
import time
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

import warnings
warnings.filterwarnings("ignore")

In [4]:
# ============================================================
# 1. LOAD DATASET & DATA CONCATENATION
# ============================================================
for dirname, _, filenames in os.walk('/kaggle/input/datasets/khinguyn112/dataset-cic-ids2017'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

DATA_DIR = '/kaggle/input/datasets/khinguyn112/dataset-cic-ids2017'
files = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.csv'), recursive=True))

dfs = []
for f in files:
    df = pd.read_csv(f, encoding='latin1', low_memory=False)
    df.columns = df.columns.str.strip()
    print(f"{os.path.basename(f):60s} {df.shape}")
    dfs.append(df)

data = pd.concat(dfs, axis=0, ignore_index=True)
del dfs

print("\nShape sau khi nối:", data.shape)
print(data['Label'].value_counts())


ValueError: No objects to concatenate

In [ ]:
# ============================================================
# 2. DATA TYPE FIX
# ============================================================
# Sửa nhãn bị lỗi ký tự
data['Label'] = data['Label'].str.replace('ï¿½', '-', regex=False).str.strip()
print(data['Label'].unique())

# Kiểm tra kiểu dữ liệu
print(data.dtypes.value_counts())
print("Cột không phải số:", data.select_dtypes(exclude='number').columns.tolist())

# Kiểm tra inf và NaN
num_cols = data.select_dtypes('number').columns
inf_count = np.isinf(data[num_cols]).sum()
print("Cột có inf:\n", inf_count[inf_count > 0])
print("Cột có NaN:\n", data.isna().sum()[lambda s: s > 0])

# Thay inf bằng NaN rồi bỏ các dòng NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)
before = len(data)
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)
print(f"Đã bỏ {before - len(data)} dòng, còn lại {data.shape}")

In [ ]:
# ============================================================
# 3. DATA CLEANING (downcast, giá trị âm, trùng lặp, cột hằng số/trùng)
# ============================================================
# Downcast giảm RAM
for col in data.select_dtypes('float64').columns:
    data[col] = pd.to_numeric(data[col], downcast='float')
for col in data.select_dtypes('int64').columns:
data[col] = pd.to_numeric(data[col], downcast='integer')
print(f"[1] RAM: {data.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

# Bỏ dòng có giá trị âm bất thường (giữ Init_Win_bytes_* vì -1 là hợp lệ)
feat_cols = [c for c in data.columns if c != 'Label']
neg = (data[feat_cols] < 0).sum()
check_cols = [c for c in neg[neg > 0].index if not c.startswith('Init_Win_bytes')]
print("[2] Cột có giá trị âm bất thường:", check_cols)

before = data['Label'].value_counts()
data = data.loc[~(data[check_cols] < 0).any(axis=1)].reset_index(drop=True)
after = data['Label'].value_counts()
print(pd.DataFrame({'before': before, 'after': after, 'removed': before - after}))

# Bỏ dòng trùng
before = data['Label'].value_counts()
data = data.drop_duplicates().reset_index(drop=True)
after = data['Label'].value_counts()
print("\n[3] Bỏ dòng trùng:")
print(pd.DataFrame({'before': before, 'after': after, 'removed': before - after}))

# Bỏ cột hằng số
feat_cols = [c for c in data.columns if c != 'Label']
constant_cols = [c for c in feat_cols if data[c].nunique() <= 1]
data = data.drop(columns=constant_cols)
print(f"\n[4] Cột hằng số đã bỏ ({len(constant_cols)}):", constant_cols)

# Bỏ cột trùng lặp
feat_cols = [c for c in data.columns if c != 'Label']
seen, dup_cols = {}, []
for c in feat_cols:
    key = hash(data[c].values.tobytes())
    if key in seen and data[c].equals(data[seen[key]]):
        dup_cols.append(c)
    else:
        seen[key] = c
data = data.drop(columns=dup_cols)
print(f"[5] Cột trùng đã bỏ ({len(dup_cols)}):", dup_cols)

# Data Properties sau khi làm sạch
print("\n" + "=" * 60)
print("Shape:", data.shape)
print(f"RAM: {data.memory_usage(deep=True).sum() / 1024**2:.0f} MB")
print("\nKiểu dữ liệu:\n", data.dtypes.value_counts())
print("\nSố NaN còn lại:", data.isna().sum().sum())

label_dist = data['Label'].value_counts()
label_df = pd.DataFrame({'count': label_dist,
                         'percent': (label_dist / len(data) * 100).round(3)})
display(label_df)
display(data.describe().T.head(10))

# Lưu lại data sau khi làm sạch
data.to_csv('/kaggle/working/data_cleaned.csv', index=False)
print("Đã lưu data_cleaned.csv")

In [ ]:
# ============================================================
# 4. TRAIN/TEST SPLIT
# ============================================================
X = data.drop(columns='Label')
y = data['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, " Test:", X_test.shape)
print(y_train.value_counts())

In [ ]:
# ============================================================
# 5. DROP COLUMN BASED ON CORRELATION
# ============================================================
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

threshold = 0.95
to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

print(f"Số cột bị bỏ do tương quan cao (>{threshold}): {len(to_drop)}")
print(to_drop)

X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

print("\nShape sau khi bỏ cột tương quan cao:")
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
# ============================================================
# 6. FEATURE SCALING
# ============================================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(X_train_scaled.describe().T[['mean', 'std']].head())

In [ ]:
# ============================================================
# 7. TRAIN 4 MODEL
# ============================================================
models = {
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), random_state=42)
}

trained_models = {}

for name, model in models.items():
    start = time.time()
    model.fit(X_train_scaled, y_train)
    elapsed = time.time() - start
    trained_models[name] = model
    print(f"{name}: train xong trong {elapsed:.1f}s")

# Lưu lại model, scaler, tập test
with open('/kaggle/working/trained_models.pkl', 'wb') as f:
    pickle.dump(trained_models, f)
with open('/kaggle/working/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
X_test_scaled.to_csv('/kaggle/working/X_test_scaled.csv', index=False)
y_test.to_csv('/kaggle/working/y_test.csv', index=False)
print("Đã lưu trained_models.pkl, scaler.pkl, X_test_scaled.csv, y_test.csv")

# Kiểm tra ngay lập tức
for fn in ['trained_models.pkl', 'scaler.pkl', 'X_test_scaled.csv', 'y_test.csv']:
    path = f'/kaggle/working/{fn}'
    print(fn, '-', os.path.exists(path))


In [ ]:
# ============================================================
# 8. ĐÁNH GIÁ MÔ HÌNH
# ============================================================

# 8.1 Accuracy, Precision, Recall, F1-score (đa lớp)
results = []
for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    prec_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision (macro)': prec_macro,
        'Recall (macro)': rec_macro,
        'F1 (macro)': f1_macro,
        'Precision (weighted)': prec_weighted,
        'Recall (weighted)': rec_weighted,
        'F1 (weighted)': f1_weighted
    })

results_df = pd.DataFrame(results).set_index('Model')
display(results_df.round(4))


# 8.2 Classification Report chi tiết theo từng lớp
for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    print(f"\n{'='*25} {name} {'='*25}")
    print(classification_report(y_test, y_pred, zero_division=0))


# 8.3 Confusion Matrix (đa lớp - 15 lớp)
labels = sorted(y_test.unique())

fig, axes = plt.subplots(2, 2, figsize=(20, 16))

for ax, (name, model) in zip(axes.ravel(), trained_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax, cbar=False)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=90)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()


# 8.4 Đánh giá nhị phân: Bình thường (Normal) vs Bất thường (Attack)

# Confusion Matrix nhị phân
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, (name, model) in zip(axes.ravel(), trained_models.items()):
    y_pred = model.predict(X_test_scaled)
    
    y_test_bin = y_test.apply(lambda x: 'Bình thường' if x == 'BENIGN' else 'Bất thường')
    y_pred_bin = pd.Series(y_pred, index=y_test.index).apply(
        lambda x: 'Bình thường' if x == 'BENIGN' else 'Bất thường')
    
    cm = confusion_matrix(y_test_bin, y_pred_bin, labels=['Bình thường', 'Bất thường'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Bình thường', 'Bất thường'],
                yticklabels=['Bình thường', 'Bất thường'],
                ax=ax, cbar=False, annot_kws={'size': 14})
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('Dự đoán')
    ax.set_ylabel('Thực tế')

plt.suptitle('Phát hiện Bình thường vs Bất thường (4 model)', fontsize=15)
plt.tight_layout()
plt.show()

# Bảng tóm tắt số liệu nhị phân
summary = []
for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    y_test_bin = y_test.apply(lambda x: 0 if x == 'BENIGN' else 1)
    y_pred_bin = pd.Series(y_pred, index=y_test.index).apply(lambda x: 0 if x == 'BENIGN' else 1)
    
    acc = accuracy_score(y_test_bin, y_pred_bin)
    prec = precision_score(y_test_bin, y_pred_bin)
    rec = recall_score(y_test_bin, y_pred_bin)
    f1 = f1_score(y_test_bin, y_pred_bin)
    
    summary.append({'Model': name, 'Accuracy': acc, 'Precision': prec,
                     'Recall (phát hiện tấn công)': rec, 'F1': f1})

summary_df = pd.DataFrame(summary).set_index('Model')
display(summary_df.round(4))